# Day 1-02｜Homography 與單點投影
> Python 籃球運動資料分析課程  
> 使用相機畫面與鳥瞰圖的對應點，建立平面投影轉換。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 理解 Homography 需要至少四組對應點。
- 計算相機畫面到 BEV 的轉換矩陣。
- 將一個球員腳底點投影到鳥瞰圖並判讀位置。

## 完成產出
- 相機畫面與 BEV 的對照圖，以及一個投影後的座標點。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 載入範例對應點。
2. 計算 Homography 並檢查參考點。
3. 修改測試點，觀察 BEV 位置變化。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


In [ ]:
import numpy as np
from src.cv_utils import (
    load_json,
    read_image_rgb,
    draw_points,
    show_image,
    side_by_side,
    save_image_rgb,
)
from src.geometry_utils import compute_homography, project_points

points_json = COURSE_ROOT / "assets" / "samples" / "sample_homography_points.json"
data = load_json(points_json)

camera_points = data["camera_points"]
bev_points = data["bev_points"]

H = compute_homography(camera_points, bev_points)
print("Homography matrix H =")
print(np.round(H, 3))

In [ ]:
frame = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png")
bev = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_bev_court.png")

frame_vis = draw_points(
    frame, camera_points, labels=[str(i) for i in range(len(camera_points))]
)
bev_vis = draw_points(bev, bev_points, labels=[str(i) for i in range(len(bev_points))])
show_image(side_by_side(frame_vis, bev_vis), "左：相機點；右：BEV 對應點")

接著指定相機畫面中的任一點，例如球員腳底點，看看它在 BEV 上的位置。

In [ ]:
# 你可以修改這個點。
camera_point = [[586, 715]]
bev_point = project_points(camera_point, H)

print("camera point:", camera_point[0])
print("BEV point:", np.round(bev_point[0], 2).tolist())

frame_vis = draw_points(frame, camera_point, labels=["input"])
bev_vis = draw_points(bev, bev_point, labels=["projected"])
out = side_by_side(frame_vis, bev_vis)
show_image(out, "單點投影結果")

save_path = COURSE_ROOT / "assets" / "results" / "d1_02_point_projection.png"
save_image_rgb(save_path, out)
print("saved:", save_path)

課堂操作：將 `camera_point` 改為另一名球員的腳底位置，記錄 BEV 投影點的變化。